# Lab 10 — Perpetual Funding: Headline Rate vs. Realized Carry

**Goal:** compare the funding rate visible **right now** with the funding that was actually paid over the **last week**.

### Flow
1. Pull Hyperliquid perpetuals with **more than $1M of 24h volume**.
2. Show the current hourly funding rate and annualize it.
3. Hand-pick **2–3 tokens** that look interesting.
4. Pull one week of historical hourly funding for those tokens.
5. Compare the current annualized headline rate with the historically realized rate.
6. Subtract a simple trading-fee hurdle.

The key question is:

> **How different is the yield implied by one noisy funding observation from the carry a trader actually realized over time?**

**Expected time:** 20–30 minutes.


In [ ]:
import time
import pandas as pd
import requests

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

INFO_URL = "https://api.hyperliquid.xyz/info"

HOURS_PER_YEAR = 24 * 365
MIN_VOLUME_USD = 1_000_000
NOTIONAL_USD = 10_000

# Standardized Tier-0 taker-fee benchmark:
# spot 7 bps + perp 4.5 bps, on both entry and exit.
ROUND_TRIP_FEE_BPS = 2 * (7.0 + 4.5)   # 23 bps
ROUND_TRIP_FEE_USD = NOTIONAL_USD * ROUND_TRIP_FEE_BPS / 10_000

def post_info(payload):
    r = requests.post(INFO_URL, json=payload, timeout=20)
    r.raise_for_status()
    return r.json()


In [ ]:
# Pull all perpetual markets and keep only tokens with > $1M of 24h notional volume.
meta, contexts = post_info({"type": "metaAndAssetCtxs"})

if len(meta["universe"]) != len(contexts):
    raise ValueError("Hyperliquid metadata and market-context arrays are not aligned.")

rows = []
for asset, ctx in zip(meta["universe"], contexts):
    if asset.get("isDelisted", False):
        continue

    volume = float(ctx["dayNtlVlm"])
    if volume <= MIN_VOLUME_USD:
        continue

    funding_hourly = float(ctx["funding"])
    annualized_rate = funding_hourly * HOURS_PER_YEAR

    rows.append({
        "coin": asset["name"],
        "24h_volume_$": volume,
        "current_hourly_funding_%": 100 * funding_hourly,
        "current_annualized_funding_%": 100 * annualized_rate,
        "annualized_gross_profit_on_$10k": NOTIONAL_USD * annualized_rate,
    })

current_markets = (
    pd.DataFrame(rows)
    .sort_values("current_annualized_funding_%", ascending=False)
    .reset_index(drop=True)
)

print(f"{len(current_markets)} perpetuals have more than $1M of 24h volume.")
display(current_markets)


## TODO 1 — Pick tokens

From the table above, choose **2 or 3 tokens** whose funding you want to investigate.

There is no scoring rule. Pick whatever looks interesting — for example, a token with a very high current rate and one or two familiar large markets.


In [ ]:
# TODO: replace these comments with 2 or 3 token symbols from the table above.
CHOSEN_COINS = [
    # "ETH",
    # "BTC",
]

if not 2 <= len(CHOSEN_COINS) <= 3:
    raise ValueError("Choose 2 or 3 tokens.")

missing = set(CHOSEN_COINS) - set(current_markets["coin"])
if missing:
    raise ValueError(f"Not in the >$1M-volume table: {sorted(missing)}")

print("Chosen tokens:", CHOSEN_COINS)


In [ ]:
def fetch_last_week_funding(coin):
    end_ms = int(time.time() * 1000)
    start_ms = end_ms - 7 * 24 * 60 * 60 * 1000

    raw = post_info({
        "type": "fundingHistory",
        "coin": coin,
        "startTime": start_ms,
        "endTime": end_ms,
    })

    h = pd.DataFrame(raw)
    if h.empty:
        raise ValueError(f"No funding history returned for {coin}.")

    h["fundingRate"] = h["fundingRate"].astype(float)
    h["time"] = pd.to_datetime(h["time"], unit="ms", utc=True)
    return h.sort_values("time").reset_index(drop=True)

current_lookup = current_markets.set_index("coin")
summary_rows = []
funding_history = {}

for coin in CHOSEN_COINS:
    h = fetch_last_week_funding(coin)
    funding_history[coin] = h

    # Sum of hourly funding actually paid during the observed week.
    realized_return = h["fundingRate"].sum()

    # Annualize the average hourly rate only so it is directly comparable
    # with the instantaneous annualized headline number.
    realized_annualized = h["fundingRate"].mean() * HOURS_PER_YEAR

    gross_week_dollars = NOTIONAL_USD * realized_return

    summary_rows.append({
        "coin": coin,
        "hours_observed": len(h),
        "current_annualized_%": current_lookup.loc[coin, "current_annualized_funding_%"],
        "realized_last_week_%": 100 * realized_return,
        "realized_annualized_%": 100 * realized_annualized,
        "gross_funding_on_$10k": gross_week_dollars,
        "round_trip_fee_$": ROUND_TRIP_FEE_USD,
        "net_after_fee_on_$10k": gross_week_dollars - ROUND_TRIP_FEE_USD,
    })

comparison = pd.DataFrame(summary_rows).set_index("coin")
display(comparison)


## TODO 2 — What changed when you looked backward?

Using the table above, answer in **3–5 sentences**:

1. Which chosen token has the highest **current annualized** funding rate?
2. Which has the highest **realized annualized** funding rate over the last week?
3. Is the current headline rate a good description of what traders actually earned last week?
4. After the standardized **23 bps round-trip fee**, would last week's carry have been profitable on a $10,000 position?

Submit the answers in the submission form.

BONUS (TODO 3): try different lengths of time by changing the parameter in the last cell to see how they perform across different lengths of time/market conditions.

## Takeaway

The live funding screen tells you **where the imbalance is right now**. It does not tell you what you will actually earn.

The simple workflow is:

**current funding → annualize the headline → inspect 1-week history → measure realized carry → subtract fees**

The annualized historical number is included only to make the comparison easy. The actual one-week return is the column `realized_last_week_%`.

**Fee note:** the 23 bps figure is a standardized Tier-0 Hyperliquid taker benchmark for a spot/perp round trip. If the spot leg is executed elsewhere, the actual spot fee will differ.
